In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

def pan_tompkins_preprocess(ecg_signal, fs=200):
    """
    Applies the classical Pan-Tompkins preprocessing pipeline:
    Cascaded Bandpass Filter -> Derivative -> Squaring -> Moving Window Integration (MWI)
    """
    # 1. Pan-Tompkins specific integer low-pass & high-pass filter (optimized for fs=200Hz)
    b_lp = np.zeros(13); b_lp[0], b_lp[6], b_lp[12] = 1, -2, 1
    a_lp = [1, -2, 1]
    lp_signal = signal.lfilter(b_lp, a_lp, ecg_signal)
    
    b_hp = np.zeros(33); b_hp[0], b_hp[16], b_hp[17], b_hp[32] = -1/32, 1, -1, 1/32
    a_hp = [1, -1]
    hp_signal = signal.lfilter(b_hp, a_hp, lp_signal)
    
    # 2. Derivative stage
    b_der = [2/8, 1/8, 0, -1/8, -2/8]
    der_signal = signal.lfilter(b_der, 1, hp_signal)
    
    # 3. Squaring stage
    sq_signal = der_signal ** 2
    
    # 4. Moving Window Integration (Window size approx 150ms)
    window_len = int(0.15 * fs)
    b_mwi = np.ones(window_len) / window_len
    mwi_signal = signal.lfilter(b_mwi, 1, sq_signal)
    
    return mwi_signal

def hilbert_envelope_preprocess(filtered_signal, fs=200):
    """
    Computes the magnitude of the analytic signal via Hilbert Transform,
    then applies a smoothing window to isolate the QRS envelope.
    """
    # Compute the analytical signal amplitude envelope
    analytic_signal = signal.hilbert(filtered_signal)
    amplitude_envelope = np.abs(analytic_signal)
    
    # Smooth the envelope using a moving average window
    window_len = int(0.15 * fs)
    b_mwi = np.ones(window_len) / window_len
    smoothed_envelope = signal.lfilter(b_mwi, 1, amplitude_envelope)
    
    return smoothed_envelope

def shannon_energy_preprocess(filtered_signal, fs=200):
    """
    Computes the Shannon Energy Envelope to emphasize the sharp spikes of the QRS complexes
    while suppressing lower amplitude T and P waves.
    """
    # Normalize the filtered signal intensity to a max of 1.0
    norm_signal = filtered_signal / np.max(np.abs(filtered_signal))
    
    # Avoid log(0) errors using a tiny epsilon value
    epsilon = 1e-10
    shannon_energy = - (norm_signal ** 2) * np.log(norm_signal ** 2 + epsilon)
    
    # Smooth the resulting energy wave
    window_len = int(0.15 * fs)
    b_mwi = np.ones(window_len) / window_len
    smoothed_shannon = signal.lfilter(b_mwi, 1, shannon_energy)
    
    return smoothed_shannon

def wavelet_transform_preprocess(ecg_signal, fs=200):
    """
    Applies Continuous Wavelet Transform (CWT) using a Ricker (Mexican Hat) wavelet.
    A scale width matching the nominal QRS duration (~6-10 at 200Hz) effectively highlights peaks.
    """
    # A scale width of 6 corresponds well to the typical 80-100ms QRS complex duration at 200Hz
    widths = [6]
    cwt_matrix = signal.cwt(ecg_signal, signal.ricker, widths)
    
    # Extract the scale magnitude row
    wavelet_envelope = np.abs(cwt_matrix[0])
    return wavelet_envelope


# --- MAIN EXECUTION PIPELINE ---

path = 'G:\pulse_data\\relax\\06.txt'
fs = 500  # Target sampling rate (Hz)
s = 101

original_ecg = np.loadtxt(path, delimiter='\t', usecols=1)[s*2500:(s+1)*2500]
t = np.arange(len(original_ecg)) / fs

# 0. Pre-filtering stage (5-15 Hz Butterworth bandpass) for standard envelope routines
b_band, a_band = signal.butter(3, [5.0 / (0.5 * fs), 15.0 / (0.5 * fs)], btype='band')
bandpassed_ecg = signal.filtfilt(b_band, a_band, original_ecg)

# 1. Compute Preprocessing Methods
pt_output = pan_tompkins_preprocess(original_ecg, fs=fs)
hilbert_output = hilbert_envelope_preprocess(bandpassed_ecg, fs=fs)
shannon_output = shannon_energy_preprocess(bandpassed_ecg, fs=fs)
wavelet_output = wavelet_transform_preprocess(original_ecg, fs=fs)

# 2. Visualizing and Comparing All Methods
fig, axes = plt.subplots(5, 1, figsize=(12, 11), sharex=True)

# Original Signal
axes[0].plot(t, original_ecg, color='black', lw=1.2, label='Raw ECG Signal')
axes[0].set_title('Comparison of ECG QRS-Complex Extraction Techniques', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(loc='upper right')

# Pan-Tompkins
axes[1].plot(t, pt_output, color='crimson', lw=1.5, label='Pan-Tompkins (MWI)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='upper right')

# Hilbert Envelope
axes[2].plot(t, hilbert_output, color='darkorange', lw=1.5, label='Hilbert Transform Envelope')
axes[2].set_ylabel('Amplitude')
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].legend(loc='upper right')

# Shannon Energy Envelope
axes[3].plot(t, shannon_output, color='forestgreen', lw=1.5, label='Shannon Energy Envelope')
axes[3].set_ylabel('Amplitude')
axes[3].grid(True, linestyle='--', alpha=0.6)
axes[3].legend(loc='upper right')

# Wavelet Transform
axes[4].plot(t, wavelet_output, color='dodgerblue', lw=1.5, label='CWT (Mexican Hat, Scale 6)')
axes[4].set_xlabel('Time (seconds)', fontsize=11)
axes[4].set_ylabel('Amplitude')
axes[4].grid(True, linestyle='--', alpha=0.6)
axes[4].legend(loc='upper right')

plt.tight_layout()
plt.savefig('ecg_preprocessing_comparison.png', dpi=300)